In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

## LOAD 

print(f"Loading data")
df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()

df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(   # first group by names we got 6. then we  take only OILRATENORM coloumn from that.
    lambda x:x.clip(upper=x.quantile(0.99))      # it calculated 99th percentile for this means we are capping the  outliners .
)
print(f'Loaded {len(df)} rows')


print(f"Resampling to month")
monthly_data={}
for well_name , well_df in df.groupby('NPD_WELL_BORE_NAME'):
    monthly = well_df['OIL_RATE_NORM'].resample('ME').mean().dropna()

    if (len(monthly) < 12):
        print(f'Skipping well name: {well_name} as it has less then 12 months data')
        continue

    monthly_data[well_name] = monthly
    print(f'{well_name}: {len(monthly)} months  from {monthly.min().date()} --> {monthly.max().date()}')

Loading data
Loaded 8020 rows
